# 🎬 Clip Studio (Colab)
สกิล → บท (Claude หรือวางเอง) → พากย์ ElevenLabs → คลิป MP4 ซับไทย

**ครั้งแรก:** เมนูซ้าย 🔑 *Secrets* → เพิ่ม `ELEVENLABS_API_KEY` (และ `ANTHROPIC_API_KEY` ถ้าจะให้ AI คิดบท) → เปิด *Notebook access*

**ทุกครั้ง:** Runtime → Run all แล้วกดลิงก์ `https://….gradio.live` ที่ขึ้นใน cell สุดท้าย

ข้อมูล (เทมเพลต/โปรเจกต์/คลิป) เก็บใน Google Drive: `MyDrive/ClipStudio`

**เพิ่มเทมเพลต:** แท็บ *เทมเพลต* → ลากไฟล์ .zip ของสกิลมาวาง

In [ ]:
#@title 1. ติดตั้ง + เชื่อม Google Drive
!apt-get -qq install -y fonts-thai-tlwg > /dev/null
!pip -q install gradio==6.32.0 anthropic==2.12.0 pythainlp edge-tts uharfbuzz freetype-py opencv-python-headless
import os
from google.colab import drive, userdata
drive.mount('/content/drive')
os.environ['CLIPSTUDIO_DIR'] = '/content/drive/MyDrive/ClipStudio'
for k in ('ANTHROPIC_API_KEY', 'ELEVENLABS_API_KEY'):
    try:
        os.environ[k] = userdata.get(k) or ''
        print(k, '✅' if os.environ[k] else '❌ ว่าง')
    except Exception:
        print(k, '❌ ไม่มีใน Secrets (หรือยังไม่เปิด Notebook access)')

In [ ]:
%%writefile /content/drama_adapter.py
"""เทมเพลตแบบ "ละครการ์ตูน" (engine แบบ thai-drama-script: scripts/render_ep.py + engine/)

โปรเจกต์ = 1 ซีรีส์: series.json + epNN.json + bg/ + .voice_cache/ (ใช้โครงเดียวกับที่ engine ต้องการ)
"""
from __future__ import annotations

import json
import re
import subprocess
import sys
from pathlib import Path

RUNNER = Path(__file__).with_name("drama_runner.py")
PRESETS = {"claude": "🎨 ฉากที่ Claude วาด (อนุมัติแล้ว)", "auto": "สำเร็จรูป (เดาจากชื่อ)", "market": "ตลาดสด", "rented_room": "ห้องเช่า", "garage": "อู่ซ่อมรถ"}


class DramaError(Exception):
    pass


def detect(root: Path) -> bool:
    return (root / "scripts" / "render_ep.py").is_file() and (root / "engine").is_dir()


def parse_json(text: str, what: str) -> dict:
    text = (text or "").strip()
    # รองรับกรณีวางมาพร้อม ```json ... ```
    m = re.search(r"```(?:json)?\s*(.*?)```", text, re.S)
    if m:
        text = m.group(1)
    try:
        data = json.loads(text)
    except json.JSONDecodeError as e:
        raise DramaError(f"{what}: JSON ไม่ถูกต้อง บรรทัด {e.lineno} คอลัมน์ {e.colno} ({e.msg})")
    if not isinstance(data, dict):
        raise DramaError(f"{what}: ต้องเป็น JSON object {{...}}")
    return data


def save_series(pdir: Path, text: str) -> dict:
    s = parse_json(text, "series.json")
    if not s.get("cast") or not s.get("places"):
        raise DramaError("series.json ต้องมี cast และ places")
    pdir.mkdir(parents=True, exist_ok=True)
    (pdir / "series.json").write_text(json.dumps(s, ensure_ascii=False, indent=2), "utf-8")
    return s


def save_episode(pdir: Path, text: str) -> tuple[str, dict]:
    ep = parse_json(text, "บทตอน (epNN.json)")
    if not isinstance(ep.get("ep"), int) or not ep.get("shots"):
        raise DramaError("บทตอนต้องมี \"ep\" (ตัวเลข) และ \"shots\"")
    name = f"ep{ep['ep']:02d}.json"
    (pdir / name).write_text(json.dumps(ep, ensure_ascii=False, indent=2), "utf-8")
    return name, ep


def episodes(pdir: Path) -> list[str]:
    return sorted(p.name for p in pdir.glob("ep[0-9]*.json"))


def load(pdir: Path, name: str) -> dict:
    return json.loads((pdir / name).read_text("utf-8"))


def voice_slots(series: dict) -> list[tuple[str, str, str]]:
    """ช่องเลือกเสียง: (id, ชื่อ, ประเภทเสียง)"""
    out = [(c["id"], c.get("name", c["id"]), c.get("voice", "")) for c in series.get("cast", []) if "id" in c]
    if not any(i == "hero" for i, _, _ in out):
        out.insert(0, ("hero", "ผม (ผู้เล่า)", "man"))
    out.append(("system", "เสียงระบบ", "system"))
    return out


def _run(cmd, cwd, on_line=None, timeout=3600) -> str:
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        lines.append(line.rstrip())
        if on_line:
            on_line(line.rstrip())
    p.wait(timeout=timeout)
    out = "\n".join(lines)
    if p.returncode != 0:
        raise DramaError(out[-3000:] or f"exit {p.returncode}")
    return out


def check(root: Path, pdir: Path, ep_name: str, sec: int = 0) -> tuple[bool, str]:
    cmd = [sys.executable, str(root / "scripts" / "drama_tool.py"), "check", str(pdir / ep_name)]
    if sec:
        cmd += ["--sec", str(sec)]
    try:
        out = _run(cmd, pdir)
    except DramaError as e:
        out = str(e)
    last = out.strip().splitlines()[-1] if out.strip() else ""
    return last.startswith("PASS"), out


def apply_backgrounds(pdir: Path, series: dict, choice: dict, lib_dir: Path | None = None) -> None:
    """ตั้งฉากของแต่ละสถานที่ (bg/<place>.py/.png ที่ engine อ่าน) ตามที่เลือก
    claude = ฉากที่ Claude วาดและอนุมัติแล้ว (bg/_approved) · lib:<id> = จากคลังฉาก · auto = engine เดา · อื่นๆ = ฉากสำเร็จรูป"""
    import os
    import shutil

    bg = pdir / "bg"
    bg.mkdir(exist_ok=True)
    for i, pl in enumerate(series.get("places", [])):
        pid = pl.get("id")
        if not pid or not re.fullmatch(r"[\w-]+", pid):
            continue
        key = choice.get(pid, "claude")
        py, png = bg / f"{pid}.py", bg / f"{pid}.png"
        src = None
        if key == "claude":
            src = bg / "_approved" / pid
        elif key.startswith("lib:") and lib_dir:
            src = lib_dir / key[4:] / "draw"
        if src is not None:
            if src.with_suffix(".py").exists() and src.with_suffix(".png").exists():
                shutil.copy(src.with_suffix(".py"), py)
                shutil.copy(src.with_suffix(".png"), png)
                os.utime(png)  # png ใหม่กว่า .py -> engine ใช้ภาพเลย ไม่รันโค้ดซ้ำ
            else:  # ยังไม่มีฉาก: ให้ engine ใช้ฉากสำเร็จรูปที่เดาจากชื่อ
                py.unlink(missing_ok=True)
                png.unlink(missing_ok=True)
            continue
        if key == "auto" or key not in PRESETS:
            py.unlink(missing_ok=True)
            png.unlink(missing_ok=True)
            continue
        mark = f"# clipstudio-preset: {key}\n"
        if py.exists() and py.read_text("utf-8").startswith(mark):
            continue  # ตั้งไว้แบบนี้อยู่แล้ว
        py.write_text(mark + "from scene_kit import PRESETS\n\n\ndef draw():\n"
                      f"    return PRESETS[{key!r}](seed={101 + i * 37})\n", "utf-8")
        png.unlink(missing_ok=True)


def render(root: Path, pdir: Path, ep_name: str, voice_cfg: dict, still: int = 0, on_line=None) -> Path:
    if voice_cfg.get("engine") == "eleven":  # ข้อความใน engine เขียนว่า edge-tts เสมอ แก้ให้ตรงกับที่ใช้จริง
        _cb = on_line

        def on_line(line, _cb=_cb):
            if _cb:
                _cb(line.replace("with edge-tts", "with ElevenLabs"))
    cfg = pdir / ".voice_cfg.json"
    cfg.write_text(json.dumps(voice_cfg, ensure_ascii=False), "utf-8")
    args = [str(pdir / ep_name)]
    if still:
        args += ["--still", str(still)]
    out = _run([sys.executable, str(RUNNER), str(root), str(cfg), "--"] + args, pdir, on_line)
    if voice_cfg.get("engine") == "eleven":
        out = out.replace("with edge-tts", "with ElevenLabs")
    if "ERROR" in out and "DONE" not in out and "still saved" not in out:
        raise DramaError(out[-3000:])
    if still:
        m = re.search(r"still saved: (.+\.png)", out)
        if not m:
            raise DramaError(out[-3000:])
        p = Path(m.group(1))
        return p if p.is_absolute() else pdir / p
    ep = load(pdir, ep_name)
    mp4 = pdir / f"ep{int(ep.get('ep', 0)):02d}.mp4"
    if not mp4.exists():
        raise DramaError(out[-3000:])
    if "WARNING voice failed" in out:
        line = next(x for x in out.splitlines() if "WARNING voice failed" in x)
        raise DramaError("เรนเดอร์ได้แต่ไม่มีเสียง: " + line + f"\n(ไฟล์เงียบอยู่ที่ {mp4.name})")
    return mp4


# ---------------------------------------------------------------- ฉากหลังที่ Claude ออกแบบจากบท
BG_BUILD = r"""
import sys, importlib.util
sys.path.insert(0, sys.argv[1])
spec = importlib.util.spec_from_file_location("bg", sys.argv[2]); m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
img = m.draw()
img = img[0] if isinstance(img, tuple) else img
import numpy as np, cv2
from PIL import Image
img = np.clip(np.asarray(img), 0, 255).astype(np.uint8)
if img.ndim != 3 or img.shape[2] != 3:
    raise SystemExit("draw() ต้องคืนภาพ RGB")
Image.fromarray(cv2.resize(img, (1080, 1920), interpolation=cv2.INTER_AREA)).save(sys.argv[3])
"""


def used_place_ids(pdir: Path) -> set[str]:
    ids = set()
    for name in episodes(pdir):
        for sh in load(pdir, name).get("shots", []):
            if sh.get("type") == "scene" and sh.get("place"):
                ids.add(sh["place"])
    return ids


def _bg_context(root: Path) -> str:
    parts = []
    for rel in ("reference/scenes.md", "engine/scene_kit.py", "engine/drama_sets.py"):
        f = root / rel
        if f.exists():
            parts.append(f"=== {rel} ===\n{f.read_text('utf-8')}")
    for ex in sorted(root.glob("examples/*/bg/*.py"))[:2]:
        parts.append(f"=== ตัวอย่าง {ex.relative_to(root)} ===\n{ex.read_text('utf-8')}")
    return "\n\n".join(parts)


def _place_brief(pl: dict, pdir: Path, series: dict) -> str:
    acts = []
    for name in episodes(pdir):
        for sh in load(pdir, name).get("shots", []):
            if sh.get("place") == pl["id"]:
                acts += [c.get("action", "") for c in sh.get("chars", []) if c.get("action")]
                if sh.get("sfx"):
                    acts.append("sfx: " + sh["sfx"])
    return (f"สถานที่: {json.dumps(pl, ensure_ascii=False)}\n"
            f"เรื่อง: {series.get('plot', '')[:600]}\n"
            f"สิ่งที่เกิดในสถานที่นี้ตามบท: {'; '.join(dict.fromkeys(acts))[:1500]}")


def _extract_code(text: str) -> str:
    m = re.search(r"```(?:python)?\s*(.*?)```", text, re.S)
    code = (m.group(1) if m else text).strip()
    if "def draw" not in code:
        raise DramaError("Claude ไม่ได้ส่งโค้ดฉากกลับมา")
    return code


def _build_png(root: Path, py: Path, png: Path) -> None:
    r = subprocess.run([sys.executable, "-c", BG_BUILD, str(root / "engine"), str(py), str(png)],
                       capture_output=True, text=True, timeout=180)
    if r.returncode != 0:
        raise RuntimeError((r.stderr or r.stdout)[-1500:])


def cand_paths(pdir: Path, pid: str) -> tuple[Path, Path]:
    d = pdir / "bg" / "_cand"
    return d / f"{pid}.py", d / f"{pid}.png"


def draw_candidate(root: Path, pdir: Path, series: dict, place: dict, model: str,
                   review: bool = True, feedback: str = "", log=print) -> Path:
    """ให้ Claude วาดฉากตัวเลือก (ยังไม่ใช้จริงจนกว่าจะอนุมัติ) คืน path ภาพ"""
    import base64
    import io
    import os

    import anthropic

    if not os.environ.get("ANTHROPIC_API_KEY"):
        raise DramaError("ยังไม่ได้ใส่ ANTHROPIC_API_KEY (Colab: เมนู 🔑 Secrets) จึงให้ Claude วาดฉากไม่ได้")
    client = anthropic.Anthropic()
    system = [
        {"type": "text", "text":
            "คุณวาดฉากหลังการ์ตูนแนวตั้งด้วยโค้ด Python ตามคู่มือด้านล่างเท่านั้น (ใช้ได้แค่ฟังก์ชันใน scene_kit)\n"
            "ตอบเป็นโค้ดไฟล์เดียวใน ```python``` เริ่มด้วย `from scene_kit import *` และมีฟังก์ชัน `draw()` ที่ return room.finish(...)\n"
            "ฉากต้องเป็นสถานที่ว่าง ไม่มีคน ไม่มีตัวหนังสือ เว้นพื้นตรงกลางล่างให้ตัวละครยืน แสงและบรรยากาศตรงกับเวลา/อากาศของสถานที่"},
        {"type": "text", "text": _bg_context(root), "cache_control": {"type": "ephemeral"}},
    ]
    kwargs = {"betas": ["server-side-fallback-2026-07-01"], "fallbacks": "default"} if model in ("claude-sonnet-5-5", "claude-opus-5-5") else {}

    def ask(messages):
        try:
            res = client.beta.messages.create(model=model, max_tokens=16000, output_config={"effort": "medium"},
                                              system=system, messages=messages, **kwargs)
        except anthropic.AuthenticationError:
            raise DramaError("Anthropic API key ไม่ถูกต้อง")
        except anthropic.RateLimitError:
            raise DramaError("Claude ถูกจำกัดการใช้งานชั่วคราว ลองใหม่อีกสักครู่")
        except anthropic.APIStatusError as e:
            raise DramaError(f"Claude API {e.status_code}: {e.message}")
        except anthropic.APIConnectionError:
            raise DramaError("เชื่อมต่อ Claude ไม่ได้")
        if res.stop_reason == "refusal":
            raise DramaError("Claude ปฏิเสธการวาดฉากนี้")
        return res.content, "".join(b.text for b in res.content if b.type == "text")

    py, png = cand_paths(pdir, place["id"])
    py.parent.mkdir(parents=True, exist_ok=True)
    ask_text = "วาดฉากหลังสำหรับสถานที่นี้\n" + _place_brief(place, pdir, series)
    if feedback.strip():
        prev = py.read_text("utf-8") if py.exists() else ""
        ask_text += f"\n\nผู้ใช้อยากให้ปรับ: {feedback.strip()}"
        if prev:
            ask_text += f"\n\nโค้ดฉบับก่อนหน้า (แก้จากอันนี้):\n```python\n{prev}\n```"
    log(f"🎨 Claude กำลังวาดฉาก: {place.get('name', place['id'])}")
    msgs = [{"role": "user", "content": ask_text}]
    content, last_err = None, ""
    for attempt in range(3):  # โค้ดพัง: ส่ง error กลับให้แก้ (สูงสุด 2 ครั้ง)
        content, text = ask(msgs)
        py.write_text(_extract_code(text) + "\n", "utf-8")
        try:
            _build_png(root, py, png)
            break
        except Exception as e:  # noqa: BLE001
            last_err = str(e)
            log(f"   โค้ดฉากพัง ให้ Claude แก้ (ครั้งที่ {attempt + 1})")
            msgs += [{"role": "assistant", "content": content},
                     {"role": "user", "content": f"รันแล้ว error:\n{e}\nส่งโค้ดทั้งไฟล์ที่แก้แล้ว"}]
    else:
        py.unlink(missing_ok=True)
        png.unlink(missing_ok=True)
        raise DramaError(f"Claude วาดฉากไม่สำเร็จ: {last_err[-300:]}")
    if review:  # ให้ Claude ดูภาพที่ได้ 1 รอบ แล้วแก้ถ้าไม่ตรง
        from PIL import Image
        buf = io.BytesIO()
        Image.open(png).convert("RGB").resize((540, 960)).save(buf, "JPEG", quality=80)
        msgs += [{"role": "assistant", "content": content}, {"role": "user", "content": [
            {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg",
                                         "data": base64.b64encode(buf.getvalue()).decode()}},
            {"type": "text", "text": "นี่คือภาพที่ได้ ตรงกับสถานที่ไหม (ของในฉาก แสง บรรยากาศ พื้นกลางว่าง) "
                                     "ถ้าดีแล้วตอบคำเดียวว่า OK ถ้าควรแก้ ส่งโค้ดทั้งไฟล์ที่แก้แล้ว"}]}]
        _c, text = ask(msgs)
        if "def draw" in text:
            old = py.read_text("utf-8")
            py.write_text(_extract_code(text) + "\n", "utf-8")
            try:
                _build_png(root, py, png)
                log("   Claude ดูภาพแล้วปรับให้อีกรอบ")
            except Exception:  # noqa: BLE001 — ฉบับแก้พัง ใช้ฉบับแรก
                py.write_text(old, "utf-8")
                _build_png(root, py, png)
    log(f"   ✅ วาดเสร็จ รออนุมัติ: {place.get('name', place['id'])}")
    return png


def approve_candidate(pdir: Path, place: dict, lib_dir: Path, series_title: str = "") -> str:
    """ใช้ฉากตัวเลือกกับซีรีส์นี้ + เก็บเข้าคลังฉาก คืน id ในคลัง"""
    import secrets
    import shutil
    from datetime import datetime

    py, png = cand_paths(pdir, place["id"])
    if not (py.exists() and png.exists()):
        raise DramaError("ยังไม่มีฉากที่รออนุมัติ")
    ap = pdir / "bg" / "_approved"
    ap.mkdir(parents=True, exist_ok=True)
    shutil.copy(py, ap / f"{place['id']}.py")
    shutil.copy(png, ap / f"{place['id']}.png")
    lid = secrets.token_hex(5)
    d = lib_dir / lid
    d.mkdir(parents=True)
    shutil.move(str(py), d / "draw.py")
    shutil.move(str(png), d / "draw.png")
    (d / "meta.json").write_text(json.dumps({
        "id": lid, "name": place.get("name", place["id"]), "place": place, "series": series_title,
        "createdAt": datetime.now().isoformat(timespec="seconds")}, ensure_ascii=False, indent=2), "utf-8")
    return lid


def discard_candidate(pdir: Path, pid: str) -> None:
    for f in cand_paths(pdir, pid):
        f.unlink(missing_ok=True)


def library(lib_dir: Path) -> list[dict]:
    if not lib_dir.exists():
        return []
    items = []
    for d in lib_dir.iterdir():
        m = d / "meta.json"
        if m.exists() and (d / "draw.png").exists():
            items.append(json.loads(m.read_text("utf-8")))
    return sorted(items, key=lambda x: x["createdAt"], reverse=True)


def delete_library_item(lib_dir: Path, lid: str) -> None:
    import shutil
    if re.fullmatch(r"[a-f0-9]{6,20}", lid or ""):
        shutil.rmtree(lib_dir / lid, ignore_errors=True)


def suggest_from_library(place: dict, lib: list[dict]) -> dict | None:
    """หาฉากในคลังที่ชื่อ/คำบรรยายใกล้เคียงสถานที่นี้"""
    import difflib

    def text(p):
        return f"{p.get('name', '')} {p.get('id', '')} {p.get('image_prompt', '')[:120]}".lower()

    best, score = None, 0.0
    for item in lib:
        r = difflib.SequenceMatcher(None, text(place), text(item.get("place", {}))).ratio()
        if r > score:
            best, score = item, r
    return best if score >= 0.55 else None


def bg_status(pdir: Path, series: dict, choice: dict, lib_dir: Path) -> list[dict]:
    """สถานะฉากของทุกสถานที่ที่บทใช้"""
    used = used_place_ids(pdir)
    lib = library(lib_dir)
    out = []
    for pl in series.get("places", []):
        pid = pl.get("id")
        if not pid or not re.fullmatch(r"[\w-]+", pid):
            continue
        key = choice.get(pid, "claude")
        approved = pdir / "bg" / "_approved" / f"{pid}.png"
        cand = cand_paths(pdir, pid)[1]
        if key == "claude":
            state = "approved" if approved.exists() else "missing"
            preview = approved if approved.exists() else None
        elif key.startswith("lib:"):
            p = lib_dir / key[4:] / "draw.png"
            state, preview = ("library", p) if p.exists() else ("missing", None)
        else:
            state, preview = "preset", None
        out.append({"place": pl, "key": key, "state": state, "used": pid in used,
                    "preview": str(preview) if preview else None,
                    "cand": str(cand) if cand.exists() else None,
                    "suggest": suggest_from_library(pl, lib) if state == "missing" else None})
    return out


In [ ]:
%%writefile /content/drama_runner.py
"""รัน render_ep.py ของเทมเพลตละคร (thai-drama-script) โดยไม่แก้ไฟล์เทมเพลต

ถ้าเลือก ElevenLabs จะสลับฟังก์ชันสร้างเสียง (edge-tts) เป็น ElevenLabs ตอนรันเท่านั้น
ใช้: python drama_runner.py <template_root> <voice_json> -- <args ของ render_ep.py>
voice_json: {"engine": "edge"|"eleven", "model": "...", "map": {"hero": "<voice_id>", ...}, "default": "<voice_id>"}
"""
import asyncio
import importlib.util
import json
import os
import sys
from pathlib import Path


def main():
    root, cfg_path = Path(sys.argv[1]), Path(sys.argv[2])
    rest = sys.argv[sys.argv.index("--") + 1:]
    cfg = json.loads(cfg_path.read_text("utf-8"))

    sys.path.insert(0, str(root / "engine"))
    spec = importlib.util.spec_from_file_location("render_ep", root / "scripts" / "render_ep.py")
    R = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(R)

    if cfg.get("engine") == "eleven":
        import requests

        key = os.environ.get("ELEVENLABS_API_KEY", "")
        if not key:
            sys.exit("ERROR ยังไม่ได้ใส่ ELEVENLABS_API_KEY")
        vmap, default = cfg.get("map") or {}, cfg.get("default") or ""
        model = cfg.get("model") or "eleven_v3"
        orig_voice = R.Cast.voice

        def voice(self, cid):
            hero_alias = "hero" if cid in ("narration", "hero") else cid
            vid = vmap.get(cid) or vmap.get(hero_alias) or default
            if not vid:  # ไม่ได้ตั้งเสียงให้ตัวนี้ ใช้เสียงของ hero
                vid = vmap.get("hero") or ""
            if not vid:
                raise SystemExit(f"ERROR ยังไม่ได้เลือกเสียง ElevenLabs ให้ '{cid}' (และไม่มีเสียงเริ่มต้น)")
            return ("eleven:" + vid, "", "")

        R.Cast.voice = voice
        R.Cast._orig_voice = orig_voice

        def synth(text, vid):
            r = requests.post(
                f"{os.environ.get('ELEVENLABS_BASE_URL', 'https://api.elevenlabs.io')}/v1/text-to-speech/{vid}",
                params={"output_format": "mp3_44100_128"},
                headers={"xi-api-key": key, "Content-Type": "application/json", "Accept": "audio/mpeg"},
                json={"text": text, "model_id": model},
                timeout=180,
            )
            if not r.ok:
                raise RuntimeError(f"ElevenLabs {r.status_code}: {r.text[:200]}")
            return r.content

        async def _synth_all(items, outdir):
            sem = asyncio.Semaphore(2)
            done = [0]

            async def one(it):
                p = outdir / f"{it['key']}.mp3"
                if p.exists() and p.stat().st_size > 800:  # มีแล้ว ไม่เสียเครดิตซ้ำ
                    return
                async with sem:
                    data = await asyncio.to_thread(synth, it["tts"], it["voice"].split(":", 1)[1])
                    p.write_bytes(data)
                    done[0] += 1
                    print(f"voice {done[0]}/{len(items)}", flush=True)

            await asyncio.gather(*(one(it) for it in items))

        R._synth_all = _synth_all

    sys.argv = ["render_ep.py"] + rest
    R.main()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/clip_studio.py
"""Clip Studio (Colab / Gradio)

เก็บสกิล -> ให้ Claude คิดบท หรือวางบทเอง -> แก้บทรายฉาก -> พากย์ ElevenLabs -> เรนเดอร์ MP4 (ffmpeg + ซับไทย)

ข้อมูลทั้งหมดเก็บใน CLIPSTUDIO_DIR (ใน Colab ตั้งเป็นโฟลเดอร์ใน Google Drive เพื่อไม่ให้หายตอนปิด)
API keys อ่านจาก env: ANTHROPIC_API_KEY, ELEVENLABS_API_KEY
"""
from __future__ import annotations

import json
import os
import re
import secrets
import shutil
import subprocess
import time
from datetime import datetime
from pathlib import Path

import gradio as gr
import requests
from pydantic import BaseModel, Field

import drama_adapter as D

DATA = Path(os.environ.get("CLIPSTUDIO_DIR", "clipstudio_data")).resolve()
SKILLS = DATA / "skills"
PROJECTS = DATA / "projects"
for d in (SKILLS, PROJECTS):
    d.mkdir(parents=True, exist_ok=True)

MODELS = {
    "Sonnet 5.5 (สมดุล ~1–2 บาท/บท)": "claude-sonnet-5-5",
    "Haiku 5.5 (ถูกสุด)": "claude-haiku-5-5",
    "Opus 5.5 (เก่งสุด ~2–4 บาท/บท)": "claude-opus-5-5",
}
# โมเดลที่รองรับ server-side fallback (ถ้าโดนปฏิเสธ API สลับโมเดลสำรองให้เอง)
FALLBACK_OK = {"claude-sonnet-5-5", "claude-opus-5-5"}
ELEVEN_MODELS = ["eleven_v3", "eleven_multilingual_v2", "eleven_flash_v2_5"]
SIZES = {"9:16": (1080, 1920), "16:9": (1920, 1080), "1:1": (1080, 1080)}
FPS = 30
FONT = os.environ.get("SUB_FONT", "Garuda")
FONTS_DIR = os.environ.get("SUB_FONTS_DIR", "/usr/share/fonts/truetype/tlwg")


class UserError(Exception):
    """ข้อผิดพลาดที่แสดงให้ผู้ใช้เห็นตรงๆ"""


# ---------------------------------------------------------------- storage
def _new_id() -> str:
    return secrets.token_hex(6)


def _now() -> str:
    return datetime.now().isoformat(timespec="seconds")


def _read(p: Path, default=None):
    try:
        return json.loads(p.read_text("utf-8"))
    except FileNotFoundError:
        return default


def _write(p: Path, data) -> None:
    tmp = p.with_suffix(".tmp")
    tmp.write_text(json.dumps(data, ensure_ascii=False, indent=2), "utf-8")
    tmp.replace(p)


def _safe(id_: str) -> str:
    if not re.fullmatch(r"[a-f0-9]{6,32}", id_ or ""):
        raise UserError("id ไม่ถูกต้อง")
    return id_


def list_skills() -> list[dict]:
    items = [_read(f) for f in SKILLS.glob("*.json")]
    return sorted([s for s in items if s], key=lambda s: s["updatedAt"], reverse=True)


def get_skill(sid: str | None) -> dict | None:
    return _read(SKILLS / f"{_safe(sid)}.json") if sid else None


def save_skill(sid, name, desc, content, aspect, bg, fg, voice) -> dict:
    if not (name or "").strip() or not (content or "").strip():
        raise UserError("ใส่ชื่อและเนื้อหาสกิล")
    sid = _safe(sid) if sid else _new_id()
    old = get_skill(sid) or {"createdAt": _now()}
    skill = {
        **old, "id": sid, "name": name.strip()[:120], "description": (desc or "").strip()[:500],
        "content": content,
        "style": {"aspect": aspect if aspect in SIZES else "9:16", "bgColor": bg or "#111827",
                  "textColor": fg or "#FFFFFF", "voiceId": voice or ""},
        "updatedAt": _now(),
    }
    _write(SKILLS / f"{sid}.json", skill)
    return skill


def import_skill_file(path: str) -> dict:
    text = Path(path).read_text("utf-8", errors="replace")
    name, desc = Path(path).stem, ""
    m = re.match(r"^---\r?\n(.*?)\r?\n---\r?\n?", text, re.S)
    if m:
        fm = m.group(1)
        if n := re.search(r"^name:\s*(.+)$", fm, re.M):
            name = n.group(1).strip().strip("\"'")
        if d := re.search(r"^description:\s*(.+)$", fm, re.M):
            desc = d.group(1).strip().strip("\"'")
        text = text[m.end():]
    return save_skill(None, name, desc, text, "9:16", "#111827", "#FFFFFF", "")


# ---------------------------------------------------------------- templates (zip)
# เทมเพลตแบบโฟลเดอร์/zip (มีโค้ด+ไฟล์ประกอบ) เก็บที่ templates/<id>/files/ + meta.json
# เทมเพลตแบบข้อความล้วน ใช้ระบบ skills/*.json ด้านบน (type "simple")
TEMPLATES = DATA / "templates"
TEMPLATES.mkdir(parents=True, exist_ok=True)


def _frontmatter(text: str) -> tuple[str, str, str]:
    """คืน (name, description, เนื้อหาที่ตัด frontmatter แล้ว)"""
    m = re.match(r"^---\r?\n(.*?)\r?\n---\r?\n?", text, re.S)
    if not m:
        return "", "", text
    fm = m.group(1)
    n = re.search(r"^name:\s*(.+)$", fm, re.M)
    d = re.search(r"^description:\s*(.+)$", fm, re.M)
    strip = lambda x: x.group(1).strip().strip("\"'") if x else ""  # noqa: E731
    return strip(n), strip(d), text[m.end():]


def tdir(tid: str) -> Path:
    return TEMPLATES / _safe(tid)


def template_root(t: dict) -> Path:
    return tdir(t["id"]) / "files" / t.get("root", "")


def list_templates() -> list[dict]:
    """รวมเทมเพลตทุกแบบ (simple + โฟลเดอร์)"""
    items = [{**s, "type": "simple"} for s in list_skills()]
    items += [m for m in (_read(d / "meta.json") for d in TEMPLATES.iterdir() if d.is_dir()) if m]
    return sorted(items, key=lambda t: t["updatedAt"], reverse=True)


def get_template(tid: str | None) -> dict | None:
    if not tid:
        return None
    s = get_skill(tid)
    if s:
        return {**s, "type": "simple"}
    return _read(tdir(tid) / "meta.json")


def _detect_type(root: Path) -> str:
    from drama_adapter import detect
    return "drama" if detect(root) else "files"


def _safe_extract(zip_path: str, dest: Path) -> None:
    import zipfile
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            name = info.filename
            # ชื่อไฟล์ไทยใน zip บางตัว encode เป็น cp437
            if not info.flag_bits & 0x800:
                try:
                    name = name.encode("cp437").decode("utf-8")
                except (UnicodeEncodeError, UnicodeDecodeError):
                    pass
            target = (dest / name).resolve()
            if not str(target).startswith(str(dest.resolve())) or "__MACOSX" in name:
                continue  # กัน zip-slip และไฟล์ขยะของ macOS
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src, open(target, "wb") as out:
                shutil.copyfileobj(src, out)


def _find_root(files: Path) -> Path:
    """โฟลเดอร์ที่มี SKILL.md ตื้นที่สุด หรือโฟลเดอร์ชั้นเดียวที่ห่ออยู่"""
    cands = sorted(files.rglob("SKILL.md"), key=lambda p: len(p.parts))
    if cands:
        return cands[0].parent
    kids = [p for p in files.iterdir() if not p.name.startswith(".")]
    return kids[0] if len(kids) == 1 and kids[0].is_dir() else files


def import_template_zip(path: str) -> dict:
    tid = _new_id()
    d = tdir(tid)
    files = d / "files"
    files.mkdir(parents=True)
    try:
        _safe_extract(path, files)
    except Exception as e:  # noqa: BLE001
        shutil.rmtree(d, ignore_errors=True)
        raise UserError(f"แตกไฟล์ zip ไม่ได้: {e}")
    root = _find_root(files)
    skill_md = root / "SKILL.md"
    name, desc, body = _frontmatter(skill_md.read_text("utf-8", errors="replace")) if skill_md.exists() else ("", "", "")
    ttype = _detect_type(root)
    if ttype == "files" and not any(root.rglob("*.py")):
        # มีแค่ข้อความ: เก็บเป็นเทมเพลตแบบ simple ให้แก้ในฟอร์มได้
        shutil.rmtree(d, ignore_errors=True)
        return save_skill(None, name or Path(path).stem, desc, body or "(ว่าง)", "9:16", "#111827", "#FFFFFF", "")
    meta = {"id": tid, "type": ttype, "name": name or Path(path).stem, "description": desc,
            "root": str(root.relative_to(files)) if root != files else "", "createdAt": _now(), "updatedAt": _now()}
    _write(d / "meta.json", meta)
    return meta


def import_template_file(path: str) -> dict:
    if path.lower().endswith(".zip"):
        return import_template_zip(path)
    name, desc, body = _frontmatter(Path(path).read_text("utf-8", errors="replace"))
    return save_skill(None, name or Path(path).stem, desc, body, "9:16", "#111827", "#FFFFFF", "")


def add_template_files(tid: str, paths: list[str]) -> dict:
    t = get_template(tid)
    if not t or t["type"] == "simple":
        raise UserError("เพิ่มไฟล์ได้เฉพาะเทมเพลตแบบโฟลเดอร์ (zip)")
    root = template_root(t)
    for p in paths:
        if p.lower().endswith(".zip"):
            _safe_extract(p, root)
        else:
            shutil.copy(p, root / Path(p).name)
    t["updatedAt"] = _now()
    _write(tdir(tid) / "meta.json", t)
    return t


def duplicate_template(tid: str) -> dict:
    t = get_template(tid)
    if not t:
        raise UserError("ไม่พบเทมเพลต")
    if t["type"] == "simple":
        st = t["style"]
        return save_skill(None, t["name"] + " (สำเนา)", t.get("description", ""), t["content"],
                          st["aspect"], st["bgColor"], st["textColor"], st.get("voiceId", ""))
    nid = _new_id()
    shutil.copytree(tdir(tid), tdir(nid))
    meta = {**t, "id": nid, "name": t["name"] + " (สำเนา)", "createdAt": _now(), "updatedAt": _now()}
    _write(tdir(nid) / "meta.json", meta)
    return meta


def delete_template(tid: str) -> None:
    (SKILLS / f"{_safe(tid)}.json").unlink(missing_ok=True)
    shutil.rmtree(tdir(tid), ignore_errors=True)


def export_template(tid: str) -> str:
    t = get_template(tid)
    if not t:
        raise UserError("ไม่พบเทมเพลต")
    out = Path("/tmp") / f"template_{tid}"
    if t["type"] == "simple":
        out.mkdir(exist_ok=True)
        (out / "SKILL.md").write_text(f"---\nname: {t['name']}\ndescription: {t.get('description', '')}\n---\n{t['content']}", "utf-8")
        return shutil.make_archive(str(out), "zip", out)
    return shutil.make_archive(str(out), "zip", tdir(tid) / "files")


def pdir(pid: str) -> Path:
    return PROJECTS / _safe(pid)


def list_projects(kind: str = "simple") -> list[dict]:
    items = [_read(d / "project.json") for d in PROJECTS.iterdir() if d.is_dir()]
    items = [p for p in items if p and p.get("type", "simple") == kind]
    return sorted(items, key=lambda p: p["updatedAt"], reverse=True)


def get_project(pid: str) -> dict:
    p = _read(pdir(pid) / "project.json")
    if not p:
        raise UserError("ไม่พบโปรเจกต์")
    return p


def save_project(p: dict) -> dict:
    p["updatedAt"] = _now()
    pdir(p["id"]).mkdir(parents=True, exist_ok=True)
    _write(pdir(p["id"]) / "project.json", p)
    return p


# ---------------------------------------------------------------- Claude
class Scene(BaseModel):
    narration: str = Field(description="บทพากย์ของฉากนี้ อ่านออกเสียงได้ทันที")
    caption: str = Field(description="ข้อความสั้นบนจอ (ว่างได้ ถ้าจะใช้บทพากย์เป็นซับ)")
    visual: str = Field(description="คำบรรยายภาพ/ฟุตเทจที่ควรใช้ในฉากนี้")


class Script(BaseModel):
    title: str
    scenes: list[Scene]


def _system(skill: dict | None) -> str:
    base = (
        "คุณคือคนเขียนบทคลิปวิดีโอสั้น เขียนบทที่พากย์ออกเสียงแล้วฟังลื่น เป็นธรรมชาติ\n"
        "แบ่งเป็นฉาก ฉากละ 1-3 ประโยค ยาวพอดีกับภาพหนึ่งภาพ\n"
        "เขียนภาษาเดียวกับที่ผู้ใช้ขอ (ค่าเริ่มต้น: ไทย) ตัวเลขให้เขียนเป็นคำอ่านถ้าอ่านยาก\n"
        "ห้ามใส่คำกำกับเสียง วงเล็บ หรือ emoji ในบทพากย์"
    )
    if skill:
        base += f"\n\n## สกิล/สไตล์ที่ต้องทำตาม: {skill['name']}\n{skill.get('description', '')}\n\n{skill['content']}"
    return base


def ask_claude(model: str, skill: dict | None, user_text: str) -> Script:
    import anthropic

    if not os.environ.get("ANTHROPIC_API_KEY"):
        raise UserError("ยังไม่ได้ใส่ ANTHROPIC_API_KEY (Colab: เมนู 🔑 Secrets) หรือใช้โหมด 'วางบทเอง'")
    kwargs = {}
    if model in FALLBACK_OK:
        kwargs = {"betas": ["server-side-fallback-2026-07-01"], "fallbacks": "default"}
    try:
        res = anthropic.Anthropic().beta.messages.parse(
            model=model,
            max_tokens=16000,
            output_config={"effort": "medium"},
            output_format=Script,
            system=_system(skill),
            messages=[{"role": "user", "content": user_text}],
            **kwargs,
        )
    except anthropic.AuthenticationError:
        raise UserError("Anthropic API key ไม่ถูกต้อง")
    except anthropic.RateLimitError:
        raise UserError("Claude ถูกจำกัดการใช้งานชั่วคราว ลองใหม่อีกสักครู่")
    except anthropic.APIStatusError as e:
        raise UserError(f"Claude API {e.status_code}: {e.message}")
    except anthropic.APIConnectionError:
        raise UserError("เชื่อมต่อ Claude ไม่ได้ ลองใหม่")
    if res.stop_reason == "refusal":
        why = getattr(res.stop_details, "explanation", "") if res.stop_details else ""
        raise UserError(f"Claude ปฏิเสธคำขอนี้ {why}")
    if res.stop_reason == "max_tokens":
        raise UserError("บทยาวเกิน ลองลดความยาว")
    if not res.parsed_output:
        raise UserError("Claude ตอบไม่ตรงรูปแบบ ลองใหม่")
    return res.parsed_output


def split_local(script: str) -> Script:
    parts = [t.strip() for t in re.split(r"\n\s*\n", script) if t.strip()]
    return Script(title=(parts[0][:40] if parts else "บทใหม่"),
                  scenes=[Scene(narration=t, caption="", visual="") for t in parts])


# ---------------------------------------------------------------- ElevenLabs
ELEVEN = "https://api.elevenlabs.io/v1"


def _eleven_key() -> str:
    k = os.environ.get("ELEVENLABS_API_KEY", "")
    if not k:
        raise UserError("ยังไม่ได้ใส่ ELEVENLABS_API_KEY (Colab: เมนู 🔑 Secrets)")
    return k


def _eleven_check(r: requests.Response) -> requests.Response:
    if r.ok:
        return r
    try:
        d = r.json().get("detail", r.text)
        msg = d.get("message", d) if isinstance(d, dict) else d
    except ValueError:
        msg = r.text
    raise UserError(f"ElevenLabs {r.status_code}: {msg}")


def list_voices() -> list[tuple[str, str]]:
    r = _eleven_check(requests.get(f"{ELEVEN}/voices", headers={"xi-api-key": _eleven_key()}, timeout=30))
    return [(f"{v['name']} ({v['voice_id'][:6]})", v["voice_id"]) for v in r.json()["voices"]]


def tts(text: str, voice_id: str, model_id: str) -> bytes:
    if not voice_id:
        raise UserError("ยังไม่ได้เลือกเสียง")
    r = requests.post(
        f"{ELEVEN}/text-to-speech/{voice_id}",
        params={"output_format": "mp3_44100_128"},
        headers={"xi-api-key": _eleven_key(), "Content-Type": "application/json", "Accept": "audio/mpeg"},
        json={"text": text, "model_id": model_id},
        timeout=180,
    )
    return _eleven_check(r).content


# ---------------------------------------------------------------- video
_MARKS = re.compile(r"[ัิ-ฺ็-๎]")


def vis_len(s: str) -> int:
    """ความกว้างคร่าวๆ (ไม่นับสระบน/ล่าง วรรณยุกต์)"""
    return len(_MARKS.sub("", s))


def _words(text: str) -> list[str]:
    try:
        from pythainlp.tokenize import word_tokenize
        return word_tokenize(text, keep_whitespace=True)
    except ImportError:  # ไม่มี pythainlp: ตัดทีละตัวอักษร (ไม่สวยแต่ใช้ได้)
        return list(text)


def wrap_text(text: str, max_chars: int) -> list[str]:
    lines, line = [], ""
    for w in _words(re.sub(r"\s+", " ", text).strip()):
        if line and vis_len(line + w) > max_chars:
            lines.append(line.strip())
            line = w.lstrip()
        else:
            line += w
    if line.strip():
        lines.append(line.strip())
    return lines


def _ass_time(t: float) -> str:
    cs = round(t * 100)
    return f"{cs // 360000}:{cs // 6000 % 60:02d}:{cs // 100 % 60:02d}.{cs % 100:02d}"


def _ass_color(h: str) -> str:
    h = h.lstrip("#")
    return f"&H00{h[4:6]}{h[2:4]}{h[0:2]}".upper()


def build_ass(text: str, duration: float, W: int, H: int, color: str) -> str:
    max_chars = 14 if W < H else 18 if W == H else 30
    lines = wrap_text(text, max_chars)
    chunks = [lines[i:i + 2] for i in range(0, len(lines), 2)]
    total = sum(vis_len("".join(c)) for c in chunks) or 1
    t, events = 0.0, []
    for c in chunks:
        d = vis_len("".join(c)) / total * duration
        body = "\\N".join(x.replace("\\", "＼").replace("{", "").replace("}", "") for x in c)
        events.append(f"Dialogue: 0,{_ass_time(t)},{_ass_time(t + d)},Main,,0,0,0,,{body}")
        t += d
    fs = round(min(W, H) * 0.095)
    return f"""[Script Info]
ScriptType: v4.00+
PlayResX: {W}
PlayResY: {H}
WrapStyle: 2

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Main,{FONT},{fs},{_ass_color(color)},&H00FFFFFF,&H00000000,&H80000000,1,0,0,0,100,100,0,0,1,{round(fs / 12)},2,2,60,60,{round(H * 0.14)},1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
""" + "\n".join(events) + "\n"


def _run(cmd: list[str]) -> str:
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise UserError(f"{cmd[0]} ล้มเหลว:\n{r.stderr[-2000:]}")
    return r.stdout


def audio_duration(f: Path) -> float:
    out = _run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", str(f)])
    return float(out.strip() or 0)


def render_scene(scene: dict, d: Path, i: int, W: int, H: int, style: dict) -> Path:
    audio = d / scene["audio"] if scene.get("audio") else None
    image = d / scene["image"] if scene.get("image") else None
    text = (scene.get("caption") or scene.get("narration") or "").strip()
    alen = audio_duration(audio) if audio else 0
    dur = max(2.0, (alen or vis_len(scene.get("narration") or text) / 12) + 0.4)
    frames = int(dur * FPS + 0.999)
    ass = d / f"scene_{i}.ass"
    ass.write_text(build_ass(text, dur, W, H, style["textColor"]), "utf-8")
    out = d / f"scene_{i}.mp4"

    cmd = ["ffmpeg", "-y", "-hide_banner", "-loglevel", "error"]
    if image:
        cmd += ["-i", str(image)]
        vf = (f"scale={W*2}:{H*2}:force_original_aspect_ratio=increase,crop={W*2}:{H*2},"
              f"zoompan=z='min(zoom+0.0006,1.12)':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)'"
              f":d={frames}:s={W}x{H}:fps={FPS}")
    else:
        cmd += ["-f", "lavfi", "-i", f"color=c={style['bgColor']}:s={W}x{H}:r={FPS}:d={dur}"]
        vf = "null"
    cmd += ["-i", str(audio)] if audio else ["-f", "lavfi", "-i", "anullsrc=r=44100:cl=stereo"]
    ass_esc = str(ass).replace("\\", "/").replace(":", "\\:").replace("'", "\\'")
    vf += f",ass='{ass_esc}':fontsdir='{FONTS_DIR}',format=yuv420p"
    cmd += [
        "-filter_complex", f"[0:v]{vf}[v];[1:a]apad,aresample=44100,aformat=channel_layouts=stereo[a]",
        "-map", "[v]", "-map", "[a]", "-t", f"{dur:.2f}", "-r", str(FPS),
        "-c:v", "libx264", "-preset", "veryfast", "-crf", "20",
        "-c:a", "aac", "-b:a", "160k", "-ar", "44100", str(out),
    ]
    _run(cmd)
    return out


def render_project(p: dict, style: dict, progress=None) -> Path:
    W, H = SIZES.get(style.get("aspect"), SIZES["9:16"])
    d = pdir(p["id"])
    parts = []
    n = len(p["scenes"])
    for i, s in enumerate(p["scenes"]):
        if progress:
            progress(i / (n + 1), desc=f"เรนเดอร์ฉาก {i + 1}/{n}")
        parts.append(render_scene(s, d, i, W, H, style))
    if progress:
        progress(n / (n + 1), desc="ต่อคลิป")
    lst = d / "concat.txt"
    lst.write_text("\n".join(f"file '{x.name}'" for x in parts), "utf-8")
    out = d / "output.mp4"
    _run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-f", "concat", "-safe", "0",
          "-i", str(lst), "-c", "copy", "-movflags", "+faststart", str(out)])
    for x in parts:
        x.unlink(missing_ok=True)
    return out


# ---------------------------------------------------------------- UI helpers
TYPE_ICON = {"simple": "📝", "drama": "🎭", "files": "📦"}
TYPE_NAME = {"simple": "ข้อความ (บท+สี+ซับ)", "drama": "ละครการ์ตูน", "files": "โฟลเดอร์ (ยังไม่รองรับการเรนเดอร์)"}


def _skill_choices():
    """เฉพาะเทมเพลตข้อความ (ใช้กับโปรเจกต์แบบบท/ฉาก)"""
    return [("(ไม่ใช้เทมเพลต)", "")] + [(s["name"], s["id"]) for s in list_skills()]


def _template_choices():
    return [(f"{TYPE_ICON.get(t['type'], '📦')} {t['name']}", t["id"]) for t in list_templates()]


def _drama_choices(tid=None):
    ps = [p for p in list_projects("drama") if not tid or p.get("templateId") == tid]
    return [(f"{p['title'][:50]} · {p['updatedAt'][5:16]}", p["id"]) for p in ps]


def _project_choices():
    return [(f"{p['title'][:50]} · {len(p['scenes'])} ฉาก · {p['updatedAt'][5:16]}", p["id"]) for p in list_projects("simple")]


_voice_cache: list[tuple[str, str]] = []


def _voice_choices(current: str = ""):
    ch = [("(ค่าเริ่มต้น)", "")] + list(_voice_cache)
    if current and current not in [v for _, v in ch]:
        ch.append((current, current))
    return ch


def _df(p: dict) -> list[list[str]]:
    return [[s["narration"], s.get("caption", ""), s.get("visual", "")] for s in p["scenes"]]


def _status(p: dict) -> str:
    rows = ["| ฉาก | เสียง | ภาพ |", "|---|---|---|"]
    for i, s in enumerate(p["scenes"], 1):
        rows.append(f"| {i} | {'✅' if s.get('audio') else '—'} | {'🖼️' if s.get('image') else '—'} |")
    return "\n".join(rows)


def _rows(df) -> list[list[str]]:
    """แปลงค่าจาก gr.Dataframe (pandas หรือ list) เป็น list ของแถว"""
    if df is None:
        return []
    if hasattr(df, "values"):
        df = df.values.tolist()
    return [[("" if c is None else str(c)) for c in (list(r) + ["", "", ""])[:3]] for r in df]


def apply_table(p: dict, df) -> dict:
    """อัปเดตฉากจากตาราง: เสียงเดิมใช้ต่อได้ถ้าบทพากย์ไม่เปลี่ยน ภาพผูกกับลำดับแถว"""
    by_text = {s["narration"]: s for s in p["scenes"] if s.get("audio")}
    old = p["scenes"]
    new = []
    for i, (nar, cap, vis) in enumerate(_rows(df)):
        if not (nar.strip() or cap.strip() or vis.strip()):
            continue
        keep = by_text.get(nar)
        new.append({
            "narration": nar, "caption": cap, "visual": vis,
            "audio": keep["audio"] if keep else None, "voiceId": keep.get("voiceId") if keep else None,
            "image": old[i]["image"] if i < len(old) else None,
        })
    p["scenes"] = new
    return p


# ---------------------------------------------------------------- UI
def build_app() -> gr.Blocks:
    def err(fn):
        import functools

        @functools.wraps(fn)  # คง signature ไว้ให้ Gradio ส่ง gr.Progress เข้ามาได้
        def wrapped(*a, **k):
            try:
                return fn(*a, **k)
            except (UserError, D.DramaError) as e:
                raise gr.Error(str(e), duration=None)
        return wrapped

    with gr.Blocks(title="Clip Studio") as app:
        gr.Markdown("# 🎬 Clip Studio\nเทมเพลต → บท → เสียงพากย์ → คลิป MP4")
        pid = gr.State("")

        with gr.Tabs() as tabs:
            # ---------------- สร้างคลิป
            with gr.Tab("สร้างคลิป", id="create"):
                with gr.Row():
                    c_skill = gr.Dropdown([("(ไม่ใช้เทมเพลต)", "")] + _template_choices(), value="",
                                          label="เทมเพลต (เปลี่ยนแนวคลิปได้ที่นี่)", scale=4)
                    c_tinfo = gr.Markdown()
                with gr.Group(visible=True) as g_simple:
                    c_mode = gr.Radio(
                        [("ให้ Claude คิดเนื้อเรื่อง", "ai"), ("วางบทเอง (Claude แบ่งฉาก)", "own-ai"),
                         ("วางบทเอง (ย่อหน้าละฉาก ไม่ใช้ AI ฟรี)", "own")],
                        value="ai", label="ที่มาของบท")
                    c_model = gr.Dropdown(list(MODELS.items()), value="claude-sonnet-5-5", label="โมเดล Claude")
                    with gr.Group(visible=True) as g_ai:
                        c_topic = gr.Textbox(label="หัวข้อ / ไอเดีย", placeholder="เช่น ตำนานผีบ้านร้างริมคลอง")
                        c_dur = gr.Slider(15, 300, value=60, step=5, label="ความยาว (วินาที)")
                        c_extra = gr.Textbox(label="เพิ่มเติม (ไม่บังคับ)", lines=2)
                    c_script = gr.Textbox(label="วางบท (เว้นบรรทัดว่างระหว่างฉาก)", lines=12, visible=False)
                    c_go = gr.Button("สร้างบท →", variant="primary")

                with gr.Group(visible=False) as g_drama:
                    gr.Markdown("### 🎭 ละครการ์ตูน\n1) วาง series.json → 2) วางบทตอน epNN.json → 3) ตรวจบท → "
                                "4) เลือกเสียง → 5) เช็กฉาก ให้ Claude วาด แล้วอนุมัติ → 6) ดูภาพตัวอย่าง → เรนเดอร์")
                    d_pid = gr.State("")
                    with gr.Row():
                        d_proj = gr.Dropdown([], label="ซีรีส์ (ว่าง = สร้างใหม่จาก series.json)", scale=4)
                        d_del = gr.Button("🗑️ ลบซีรีส์", variant="stop", scale=1)
                    with gr.Accordion("1) series.json (ตัวละคร / สถานที่)", open=True) as d_acc_series:
                        d_series = gr.Code(language="json", label="วาง series.json", lines=10, max_lines=18)
                        d_series_file = gr.File(file_types=[".json"], label="หรือลากไฟล์ series.json มาวาง")
                        d_save_series = gr.Button("💾 บันทึก series.json")
                    with gr.Accordion("2) บทตอน epNN.json", open=True):
                        d_ep_text = gr.Code(language="json", label="วางบทตอน (epNN.json)", lines=10, max_lines=18)
                        d_ep_file = gr.File(file_types=[".json"], label="หรือลากไฟล์ epNN.json มาวาง")
                        with gr.Row():
                            d_save_ep = gr.Button("💾 บันทึกตอน")
                            d_ep = gr.Dropdown([], label="ตอนที่จะทำ", scale=2)
                            d_sec = gr.Number(value=90, precision=0, label="ความยาวเป้าหมาย (วินาที)")
                            d_check = gr.Button("✅ ตรวจบท")
                        d_check_out = gr.Textbox(label="ผลตรวจบท", lines=6, interactive=False)
                    with gr.Accordion("3) เสียงพากย์", open=True):
                        with gr.Row():
                            d_engine = gr.Radio([("edge-tts (ฟรี)", "edge"), ("ElevenLabs", "eleven")], value="edge",
                                                label="เสียงพากย์")
                            d_emodel = gr.Dropdown(ELEVEN_MODELS, value="eleven_v3", label="โมเดล ElevenLabs")
                        d_refresh = gr.State(0)

                        @gr.render(inputs=[d_pid, d_engine, d_refresh])
                        def _dynamic(pid_, engine, _r):
                            if not pid_ or not (pdir(pid_) / "series.json").exists():
                                gr.Markdown("_บันทึก series.json ก่อน แล้วช่องเลือกเสียง/ฉากจะขึ้นที่นี่_")
                                return
                            p = get_project(pid_)
                            series = D.load(pdir(pid_), "series.json")
                            if engine == "eleven":
                                gr.Markdown("**เสียง ElevenLabs ของแต่ละตัวละคร** (ว่าง = ใช้เสียงเริ่มต้น / เสียงผู้เล่า)")
                                vm = p.get("voices", {}).get("map", {})
                                with gr.Row():
                                    for cid, name, vt in [("_default", "เสียงเริ่มต้น", "")] + D.voice_slots(series):
                                        cur = p.get("voices", {}).get("default", "") if cid == "_default" else vm.get(cid, "")
                                        dd = gr.Dropdown(_voice_choices(cur), value=cur, allow_custom_value=True,
                                                         label=f"{name}" + (f" ({vt})" if vt else ""), min_width=180)
                                        dd.change(err(lambda v, cid=cid: _set_voice(pid_, cid, v)), dd, None)
                            else:
                                gr.Markdown("_edge-tts ใช้เสียงไทยของ Microsoft ตามประเภทเสียงใน series.json (ฟรี ต้องมีอินเทอร์เน็ต)_")
                    with gr.Accordion("4) ฉากหลัง: เช็กฉากที่ยังไม่มี → Claude วาด → ดูแล้วอนุมัติ", open=True):
                        with gr.Row():
                            d_bgmodel = gr.Dropdown(list(MODELS.items()), value="claude-sonnet-5-5",
                                                    label="โมเดล Claude ที่วาดฉาก (ต้องมี ANTHROPIC_API_KEY)")
                            d_review = gr.Checkbox(True, label="ให้ Claude ดูภาพตัวเองแล้วปรับอีกรอบก่อนโชว์ (สวยขึ้น แพงขึ้นเล็กน้อย)")

                        @gr.render(inputs=[d_pid, d_refresh])
                        def _bg_panel(pid_, r):
                            if not pid_ or not (pdir(pid_) / "series.json").exists():
                                gr.Markdown("_บันทึก series.json + บทตอนก่อน แล้วรายการฉากจะขึ้นที่นี่_")
                                return
                            p = get_project(pid_)
                            series = D.load(pdir(pid_), "series.json")
                            lib_dir = _lib_dir(p)
                            lib = D.library(lib_dir)
                            sts = D.bg_status(pdir(pid_), series, p.get("bgs", {}), lib_dir)
                            used = [x for x in sts if x["used"]]
                            missing = [x for x in used if x["state"] == "missing"]
                            pending = [x for x in sts if x["cand"]]
                            gr.Markdown(
                                f"**สถานที่ในบท {len(used)} ที่** · มีฉากแล้ว {len(used) - len(missing)} · "
                                f"⚠️ ยังไม่มี {len(missing)}" + (f" ({', '.join(x['place'].get('name', x['place']['id']) for x in missing)})" if missing else "")
                                + (f" · 🆕 รออนุมัติ {len(pending)}" if pending else "")
                                + "\n\n_ฉากที่ยังไม่มีตอนเรนเดอร์จะใช้ฉากสำเร็จรูปแทน_")
                            if missing:
                                b_all = gr.Button(f"🎨 ให้ Claude วาดฉากที่ยังไม่มีทั้งหมด ({len(missing)} ที่)", variant="primary")
                                b_all.click(err(lambda bm, rv, rr, progress=gr.Progress(): _draw_many(pid_, [x["place"] for x in missing], bm, rv, progress) or rr + 1),
                                            [d_bgmodel, d_review, d_refresh], d_refresh)
                            src = ([("🎨 ฉากที่ Claude วาด (อนุมัติแล้ว)", "claude")]
                                   + [(f"📚 {it['name']}" + (f" · {it['series']}" if it.get("series") else ""), "lib:" + it["id"]) for it in lib]
                                   + [(v, k) for k, v in D.PRESETS.items() if k != "claude"])
                            label = {"approved": "✅ ฉากที่ Claude วาด", "library": "📚 จากคลังฉาก", "preset": "🧱 ฉากสำเร็จรูป",
                                     "missing": "⚠️ ยังไม่มีฉาก"}
                            def _row(st):
                                pl = st["place"]
                                plid = pl["id"]
                                with gr.Group():
                                    gr.Markdown(f"#### {pl.get('name', plid)} · {label[st['state']]}"
                                                + ("" if st["used"] else " · _(บทตอนที่มียังไม่ใช้)_"))
                                    with gr.Row():
                                        if st["preview"]:
                                            gr.Image(st["preview"], label="ฉากที่ใช้อยู่", height=320, interactive=False)
                                        if st["cand"]:
                                            gr.Image(st["cand"], label="🆕 Claude วาดใหม่ (รออนุมัติ)", height=320, interactive=False)
                                        with gr.Column():
                                            dd = gr.Dropdown(src, value=st["key"], label="ใช้ฉากจาก")
                                            dd.change(err(lambda v, rr, plid=plid: _set_bg(pid_, plid, v) or rr + 1), [dd, d_refresh], d_refresh)
                                            if st["suggest"]:
                                                sg = st["suggest"]
                                                b_sg = gr.Button(f"💡 ใช้ฉากในคลังที่คล้ายกัน: {sg['name']}")
                                                b_sg.click(err(lambda rr, plid=plid, lid=sg["id"]: _set_bg(pid_, plid, "lib:" + lid) or rr + 1),
                                                           d_refresh, d_refresh)
                                            fb = gr.Textbox(label="อยากให้ปรับอะไร (ไม่บังคับ)", placeholder="เช่น ให้เป็นกลางคืน มีโต๊ะพับสีฟ้า")
                                            b_draw = gr.Button("🔁 ให้ Claude วาดใหม่" if (st["cand"] or st["preview"]) else "🎨 ให้ Claude วาด")
                                            b_draw.click(err(lambda f, bm, rv, rr, plid=plid, progress=gr.Progress():
                                                             _draw_one(pid_, plid, bm, rv, f, progress) or rr + 1),
                                                         [fb, d_bgmodel, d_review, d_refresh], d_refresh)
                                            if st["cand"]:
                                                b_ok = gr.Button("✅ ใช้ฉากนี้ + เก็บเข้าคลัง", variant="primary")
                                                b_ok.click(err(lambda rr, plid=plid: _approve(pid_, plid) or rr + 1), d_refresh, d_refresh)
                                                b_no = gr.Button("❌ ไม่เอา")
                                                b_no.click(err(lambda rr, plid=plid: D.discard_candidate(pdir(pid_), plid) or rr + 1),
                                                           d_refresh, d_refresh)

                            for st in [x for x in sts if x["used"]]:
                                _row(st)
                            unused = [x for x in sts if not x["used"]]
                            if unused:
                                with gr.Accordion(f"สถานที่ที่บทยังไม่ใช้ ({len(unused)})", open=False):
                                    for st in unused:
                                        _row(st)
                    with gr.Accordion("5) ภาพตัวอย่าง & เรนเดอร์", open=True):
                        with gr.Row():
                            d_still_n = gr.Number(value=1, precision=0, minimum=1, label="ช็อตที่")
                            d_still = gr.Button("🖼️ ดูภาพตัวอย่างช็อตนี้")
                            d_render = gr.Button("🎬 เรนเดอร์ทั้งตอน", variant="primary")
                            d_force = gr.Checkbox(False, label="เรนเดอร์แม้บทยังไม่ผ่านการตรวจ (ทดลอง)")
                        d_img = gr.Image(label="ภาพตัวอย่าง", height=560, interactive=False)
                        d_video = gr.Video(label="คลิป", height=640)
                        d_log = gr.Textbox(label="บันทึกการทำงาน", lines=6, interactive=False)

            # ---------------- แก้บท / พากย์ / เรนเดอร์
            with gr.Tab("แก้บท (แบบข้อความ)", id="edit"):
                with gr.Row():
                    e_proj = gr.Dropdown(_project_choices(), label="โปรเจกต์", scale=4)
                    e_reload = gr.Button("🔄", scale=0, min_width=50)
                    e_del = gr.Button("🗑️ ลบโปรเจกต์", variant="stop", scale=1)
                e_title = gr.Textbox(label="ชื่อคลิป")
                with gr.Row():
                    e_skill = gr.Dropdown(_skill_choices(), label="เทมเพลต (สี/ซับ/สัดส่วน)")
                    e_voice = gr.Dropdown(_voice_choices(), label="เสียงพากย์", allow_custom_value=True)
                    e_emodel = gr.Dropdown(ELEVEN_MODELS, value="eleven_v3", label="โมเดลเสียง")
                e_table = gr.Dataframe(headers=["บทพากย์", "ข้อความบนจอ (ว่าง=ใช้บทพากย์)", "ภาพที่ควรใช้"],
                                       datatype=["str", "str", "str"], column_count=(3, "fixed"),
                                       interactive=True, wrap=True, label="ฉาก (แก้ในตารางได้ เพิ่มแถว = เพิ่มฉาก)")
                with gr.Row():
                    e_save = gr.Button("💾 บันทึกบท")
                    e_tts = gr.Button("🎙️ พากย์ทุกฉาก", variant="primary")
                    e_render = gr.Button("🎬 เรนเดอร์คลิป", variant="primary")
                with gr.Row():
                    with gr.Column():
                        e_status = gr.Markdown()
                    with gr.Column():
                        e_scene = gr.Number(value=1, precision=0, minimum=1, label="ฉากที่")
                        e_img = gr.Image(type="filepath", label="ภาพประกอบฉากนี้ (อัปโหลดแล้วบันทึกอัตโนมัติ)")
                        e_audio = gr.Audio(label="เสียงฉากนี้", interactive=False)
                        with gr.Row():
                            e_tts1 = gr.Button("🎙️ พากย์ฉากนี้ใหม่")
                            e_rmimg = gr.Button("ลบภาพ")
                e_video = gr.Video(label="คลิป", height=640)

            # ---------------- เทมเพลต
            with gr.Tab("เทมเพลต", id="skills"):
                s_files = gr.File(file_count="multiple", file_types=[".zip", ".md", ".txt", ".markdown"],
                                  label="➕ เพิ่มเทมเพลต: ลาก .zip (ทั้งโฟลเดอร์สกิล) หรือ .md/.txt มาวาง ได้หลายไฟล์")
                s_pick = gr.Dropdown(_template_choices(), label="เลือกเทมเพลต (ว่าง = สร้างเทมเพลตข้อความใหม่)")
                s_info = gr.Markdown()
                with gr.Row():
                    s_dup = gr.Button("📄 ทำสำเนา")
                    s_export = gr.Button("⬇️ ดาวน์โหลด .zip")
                    s_del = gr.Button("🗑️ ลบเทมเพลต", variant="stop")
                s_zip = gr.File(label="ไฟล์ดาวน์โหลด", visible=False)
                with gr.Group(visible=False) as g_tfiles:
                    s_add = gr.File(file_count="multiple", label="เพิ่ม/แทนที่ไฟล์ในเทมเพลตนี้ (ไฟล์เดี่ยวหรือ .zip)")
                with gr.Group(visible=True) as g_tform:
                    s_name = gr.Textbox(label="ชื่อเทมเพลต")
                    s_desc = gr.Textbox(label="คำอธิบายสั้นๆ")
                    s_content = gr.Textbox(label="คำสั่ง/สไตล์ให้ Claude ทำตาม", lines=12)
                    with gr.Row():
                        s_aspect = gr.Dropdown(list(SIZES), value="9:16", label="สัดส่วนคลิป")
                        s_bg = gr.ColorPicker("#111827", label="สีพื้น")
                        s_fg = gr.ColorPicker("#FFFFFF", label="สีซับ")
                        s_voice = gr.Dropdown(_voice_choices(), value="", label="เสียงพากย์", allow_custom_value=True)
                    s_save = gr.Button("💾 บันทึกเทมเพลต", variant="primary")

            # ---------------- ตั้งค่า
            with gr.Tab("ตั้งค่า", id="settings"):
                gr.Markdown(
                    f"ข้อมูลเก็บที่ `{DATA}`\n\n"
                    f"- Anthropic key: {'✅' if os.environ.get('ANTHROPIC_API_KEY') else '❌ ไม่มี (ใช้ได้แค่โหมดวางบทเอง)'}\n"
                    f"- ElevenLabs key: {'✅' if os.environ.get('ELEVENLABS_API_KEY') else '❌ ไม่มี'}\n\n"
                    "ใส่ key ที่เมนู 🔑 Secrets ของ Colab แล้วรัน cell ใหม่")
                st_load = gr.Button("โหลดรายชื่อเสียง ElevenLabs")
                st_out = gr.Markdown()

        # ===== events: create
        c_mode.change(lambda m: (gr.update(visible=m == "ai"), gr.update(visible=m != "ai"),
                                 gr.update(visible=m != "own")),
                      c_mode, [g_ai, c_script, c_model])

        @err
        def create(skill_id, mode, model, topic, dur, extra, script):
            skill = get_skill(skill_id)
            if mode == "ai":
                if not topic.strip():
                    raise UserError("ใส่หัวข้อก่อน")
                sc = ask_claude(model, skill, f"หัวข้อ/ไอเดีย: {topic}\nความยาวคลิปประมาณ {int(dur)} วินาที\n"
                                              + (f"เพิ่มเติม: {extra}\n" if extra.strip() else "") + "เขียนบทให้ครบทุกฉาก")
            else:
                if not script.strip():
                    raise UserError("วางบทก่อน")
                sc = split_local(script) if mode == "own" else ask_claude(
                    model, skill, "นี่คือบทที่ผู้ใช้เขียนเอง ให้แบ่งเป็นฉาก ใส่ caption และ visual ให้แต่ละฉาก\n"
                    "ห้ามแก้ไข เพิ่ม หรือตัดคำในบทพากย์ narration ต้องเป็นข้อความเดิมเรียงต่อกันครบทุกคำ\n\n" + script)
            p = save_project({
                "id": _new_id(), "createdAt": _now(), "title": sc.title, "skillId": skill_id or None,
                "voiceId": (skill or {}).get("style", {}).get("voiceId", ""),
                "scenes": [{**s.model_dump(), "audio": None, "voiceId": None, "image": None} for s in sc.scenes],
            })
            gr.Info("ได้บทแล้ว ตรวจ/แก้ได้เลย")
            return gr.update(choices=_project_choices(), value=p["id"]), gr.Tabs(selected="edit")

        c_go.click(create, [c_skill, c_mode, c_model, c_topic, c_dur, c_extra, c_script], [e_proj, tabs])

        # ===== events: editor
        def load(pid_):
            if not pid_:
                return "", "", gr.update(), gr.update(), [], "", None, None, None
            p = get_project(pid_)
            out = pdir(pid_) / "output.mp4"
            return (pid_, p["title"], gr.update(choices=_skill_choices(), value=p.get("skillId") or ""),
                    gr.update(choices=_voice_choices(p.get("voiceId", "")), value=p.get("voiceId", "")),
                    _df(p), _status(p), str(out) if out.exists() else None) + _scene_view(p, 1)

        def _scene_view(p, n):
            i = int(n or 1) - 1
            if not (0 <= i < len(p["scenes"])):
                return None, None
            s, d = p["scenes"][i], pdir(p["id"])
            return (str(d / s["image"]) if s.get("image") else None,
                    str(d / s["audio"]) if s.get("audio") else None)

        load_outs = [pid, e_title, e_skill, e_voice, e_table, e_status, e_video, e_img, e_audio]
        e_proj.change(err(load), e_proj, load_outs)
        e_reload.click(lambda: gr.update(choices=_project_choices()), None, e_proj)
        e_scene.change(err(lambda pid_, n: _scene_view(get_project(pid_), n) if pid_ else (None, None)),
                       [pid, e_scene], [e_img, e_audio])

        def _save(pid_, title, skill_id, voice, df):
            if not pid_:
                raise UserError("เลือกโปรเจกต์ก่อน")
            p = get_project(pid_)
            p.update(title=title, skillId=skill_id or None, voiceId=voice or "")
            return save_project(apply_table(p, df))

        @err
        def save(pid_, title, skill_id, voice, df):
            p = _save(pid_, title, skill_id, voice, df)
            gr.Info("บันทึกแล้ว")
            return _df(p), _status(p)

        edit_in = [pid, e_title, e_skill, e_voice, e_table]
        e_save.click(save, edit_in, [e_table, e_status])

        def _target_voice(p):
            vid = p.get("voiceId") or (get_skill(p.get("skillId")) or {}).get("style", {}).get("voiceId", "")
            if not vid:
                raise UserError("เลือกเสียงพากย์ก่อน (กดโหลดรายชื่อเสียงในแท็บตั้งค่า)")
            return vid

        def _voice_scene(p, i, vid, emodel):
            s, d = p["scenes"][i], pdir(p["id"])
            name = f"voice_{i}_{int(time.time() * 1000)}.mp3"
            (d / name).write_bytes(tts(s["narration"], vid, emodel))
            if s.get("audio"):
                (d / s["audio"]).unlink(missing_ok=True)
            s.update(audio=name, voiceId=vid)

        @err
        def voice_all(pid_, title, skill_id, voice, df, emodel, n, progress=gr.Progress()):
            p = _save(pid_, title, skill_id, voice, df)
            vid = _target_voice(p)
            # ข้ามฉากที่มีเสียงของ voice นี้อยู่แล้ว (ไม่เสียเครดิตซ้ำ)
            todo = [i for i, s in enumerate(p["scenes"])
                    if s["narration"].strip() and not (s.get("audio") and s.get("voiceId") == vid)]
            for k, i in enumerate(todo):
                progress(k / max(1, len(todo)), desc=f"พากย์ฉาก {i + 1}")
                _voice_scene(p, i, vid, emodel)
                save_project(p)  # บันทึกทีละฉาก เผื่อพังกลางทาง ไม่ต้องเสียเครดิตซ้ำ
            gr.Info(f"พากย์ {len(todo)} ฉาก (ฉากที่มีเสียงแล้วข้าม)")
            return (_status(p),) + _scene_view(p, n)

        e_tts.click(voice_all, edit_in + [e_emodel, e_scene], [e_status, e_img, e_audio])

        @err
        def voice_one(pid_, title, skill_id, voice, df, emodel, n):
            p = _save(pid_, title, skill_id, voice, df)
            i = int(n or 1) - 1
            if not (0 <= i < len(p["scenes"])):
                raise UserError("ไม่มีฉากนี้")
            _voice_scene(p, i, _target_voice(p), emodel)
            save_project(p)
            return (_status(p),) + _scene_view(p, n)

        e_tts1.click(voice_one, edit_in + [e_emodel, e_scene], [e_status, e_img, e_audio])

        @err
        def set_image(pid_, n, path):
            if not pid_ or not path:
                return gr.update()
            p = get_project(pid_)
            i = int(n or 1) - 1
            if not (0 <= i < len(p["scenes"])):
                raise UserError("ไม่มีฉากนี้")
            d = pdir(pid_)
            if Path(path).parent == d:  # ภาพเดิมของฉากนี้ (ตอนโหลด) ไม่ต้องทำอะไร
                return _status(p)
            name = f"img_{i}_{int(time.time() * 1000)}{Path(path).suffix.lower() or '.jpg'}"
            shutil.copy(path, d / name)
            if p["scenes"][i].get("image"):
                (d / p["scenes"][i]["image"]).unlink(missing_ok=True)
            p["scenes"][i]["image"] = name
            save_project(p)
            return _status(p)

        e_img.upload(set_image, [pid, e_scene, e_img], e_status)

        @err
        def rm_image(pid_, n):
            p = get_project(pid_)
            i = int(n or 1) - 1
            if 0 <= i < len(p["scenes"]) and p["scenes"][i].get("image"):
                (pdir(pid_) / p["scenes"][i]["image"]).unlink(missing_ok=True)
                p["scenes"][i]["image"] = None
                save_project(p)
            return _status(p), None

        e_rmimg.click(rm_image, [pid, e_scene], [e_status, e_img])

        @err
        def render(pid_, title, skill_id, voice, df, progress=gr.Progress()):
            p = _save(pid_, title, skill_id, voice, df)
            if not p["scenes"]:
                raise UserError("ยังไม่มีฉาก")
            missing = [str(i + 1) for i, s in enumerate(p["scenes"]) if s["narration"].strip() and not s.get("audio")]
            if missing:
                gr.Warning(f"ฉาก {', '.join(missing)} ยังไม่มีเสียง จะเป็นฉากเงียบ")
            skill = get_skill(p.get("skillId"))
            style = {"aspect": "9:16", "bgColor": "#111827", "textColor": "#FFFFFF", **((skill or {}).get("style", {}))}
            out = render_project(p, style, progress)
            final = out.with_name(f"output_{int(time.time())}.mp4")  # ชื่อใหม่ กัน browser cache
            for old in pdir(pid_).glob("output_*.mp4"):
                old.unlink(missing_ok=True)
            shutil.copy(out, final)
            return str(final)

        e_render.click(render, edit_in, e_video)

        @err
        def del_project(pid_):
            if pid_:
                shutil.rmtree(pdir(pid_), ignore_errors=True)
            return gr.update(choices=_project_choices(), value=None)

        e_del.click(del_project, pid, e_proj)

        # ===== events: templates
        def _tinfo(t):
            if not t:
                return ""
            txt = f"**{TYPE_ICON.get(t['type'], '📦')} {t['name']}** · {TYPE_NAME.get(t['type'], t['type'])}"
            if t.get("description"):
                txt += f"\n\n{t['description'][:300]}"
            if t["type"] != "simple":
                root = template_root(t)
                n = sum(1 for _ in root.rglob("*") if _.is_file())
                txt += f"\n\n📁 {n} ไฟล์ · `{root.relative_to(DATA)}`"
            return txt

        def skill_form(sid):
            t = get_template(sid) if sid else None
            simple = not t or t["type"] == "simple"
            vis = (gr.update(visible=not simple), gr.update(visible=simple))
            if not t or not simple:
                return (_tinfo(t),) + vis + ("", "", "", "9:16", "#111827", "#FFFFFF", gr.update(choices=_voice_choices(), value=""))
            st = t["style"]
            return (_tinfo(t),) + vis + (t["name"], t.get("description", ""), t["content"], st["aspect"], st["bgColor"], st["textColor"],
                                         gr.update(choices=_voice_choices(st.get("voiceId", "")), value=st.get("voiceId", "")))

        skill_fields = [s_name, s_desc, s_content, s_aspect, s_bg, s_fg, s_voice]
        s_pick.change(err(skill_form), s_pick, [s_info, g_tfiles, g_tform] + skill_fields)

        def refresh_skill_dds(value=None):
            return (gr.update(choices=_template_choices(), value=value),
                    gr.update(choices=[("(ไม่ใช้เทมเพลต)", "")] + _template_choices()),
                    gr.update(choices=_skill_choices()))

        tpl_dds = [s_pick, c_skill, e_skill]

        @err
        def save_s(sid, *vals):
            t = get_template(sid) if sid else None
            if t and t["type"] != "simple":
                raise UserError("เทมเพลตแบบโฟลเดอร์แก้ในฟอร์มนี้ไม่ได้ ใช้ช่องเพิ่ม/แทนที่ไฟล์แทน")
            s_ = save_skill(sid or None, *vals)
            gr.Info("บันทึกเทมเพลตแล้ว")
            return refresh_skill_dds(s_["id"])

        s_save.click(save_s, [s_pick] + skill_fields, tpl_dds)

        @err
        def del_s(sid):
            if sid:
                delete_template(sid)
                gr.Info("ลบเทมเพลตแล้ว (โปรเจกต์ที่เคยทำไว้ยังอยู่)")
            return refresh_skill_dds("")

        s_del.click(del_s, s_pick, tpl_dds)

        @err
        def dup_s(sid):
            if not sid:
                raise UserError("เลือกเทมเพลตก่อน")
            t = duplicate_template(sid)
            gr.Info("ทำสำเนาแล้ว")
            return refresh_skill_dds(t["id"])

        s_dup.click(dup_s, s_pick, tpl_dds)

        @err
        def export_s(sid):
            if not sid:
                raise UserError("เลือกเทมเพลตก่อน")
            return gr.update(value=export_template(sid), visible=True)

        s_export.click(export_s, s_pick, s_zip)

        @err
        def upload_s(files):
            last = None
            for f in files or []:
                last = import_template_file(f if isinstance(f, str) else f.name)
            gr.Info(f"เพิ่ม {len(files or [])} เทมเพลตแล้ว")
            return refresh_skill_dds(last["id"] if last else None) + (None,)

        s_files.upload(upload_s, s_files, tpl_dds + [s_files])

        @err
        def add_files(sid, files):
            if not sid:
                raise UserError("เลือกเทมเพลตก่อน")
            t = add_template_files(sid, [f if isinstance(f, str) else f.name for f in files or []])
            gr.Info("เพิ่มไฟล์แล้ว")
            return _tinfo(t), None

        s_add.upload(add_files, [s_pick, s_add], [s_info, s_add])

        # ===== events: drama
        def _set_voice(pid_, cid, v):
            p = get_project(pid_)
            vs = p.setdefault("voices", {})
            if cid == "_default":
                vs["default"] = v or ""
            else:
                vs.setdefault("map", {})[cid] = v or ""
            save_project(p)

        def _set_bg(pid_, place_id, v):
            p = get_project(pid_)
            p.setdefault("bgs", {})[place_id] = v or "auto"
            save_project(p)

        def on_template(tid):
            t = get_template(tid)
            drama = bool(t and t["type"] == "drama")
            info = ""
            if t and t["type"] == "files":
                info = "⚠️ เทมเพลตนี้ยังไม่มีตัวเชื่อมสำหรับเรนเดอร์ ส่งให้ผู้พัฒนาเพิ่มรองรับก่อน"
            ch = _drama_choices(tid) if drama else []
            return (info, gr.update(visible=not drama), gr.update(visible=drama),
                    gr.update(choices=ch, value=ch[0][1] if ch else None))

        c_skill.change(err(on_template), c_skill, [c_tinfo, g_simple, g_drama, d_proj])

        def _drama_template(pid_):
            p = get_project(pid_)
            t = get_template(p.get("templateId"))
            if not t or t["type"] != "drama":
                raise UserError("ไม่พบเทมเพลตละครของซีรีส์นี้ (ถูกลบไปแล้ว?) เพิ่มเทมเพลตเดิมกลับเข้ามาแล้วเลือกใหม่")
            return p, t, template_root(t)

        def d_load(pid_, r):
            if not pid_:
                return "", "", gr.update(choices=[], value=None), "", r + 1, None
            d = pdir(pid_)
            eps = D.episodes(d)
            series = (d / "series.json").read_text("utf-8") if (d / "series.json").exists() else ""
            ep_txt = (d / eps[-1]).read_text("utf-8") if eps else ""
            mp4 = sorted(d.glob("ep*.mp4"))
            return (pid_, series, gr.update(choices=eps, value=eps[-1] if eps else None), ep_txt, r + 1,
                    str(mp4[-1]) if mp4 else None)

        d_proj.change(err(d_load), [d_proj, d_refresh], [d_pid, d_series, d_ep, d_ep_text, d_refresh, d_video])
        d_ep.change(err(lambda pid_, e: (pdir(pid_) / e).read_text("utf-8") if pid_ and e else gr.update()),
                    [d_pid, d_ep], d_ep_text)

        def _file_text(f):
            return Path(f if isinstance(f, str) else f.name).read_text("utf-8", errors="replace") if f else gr.update()

        d_series_file.upload(_file_text, d_series_file, d_series)
        d_ep_file.upload(_file_text, d_ep_file, d_ep_text)

        @err
        def save_series_(tid, pid_, text, r):
            t = get_template(tid)
            if not t or t["type"] != "drama":
                raise UserError("เลือกเทมเพลตละครก่อน")
            if not pid_:
                pid_ = _new_id()
                p = {"id": pid_, "type": "drama", "templateId": tid, "createdAt": _now(),
                     "voices": {"engine": "edge", "map": {}, "default": ""}, "bgs": {}}
            else:
                p = get_project(pid_)
            series = D.save_series(pdir(pid_), text)
            p["title"] = series.get("series_title") or "ซีรีส์ใหม่"
            save_project(p)
            gr.Info("บันทึก series.json แล้ว")
            return gr.update(choices=_drama_choices(tid), value=pid_), pid_, r + 1

        d_save_series.click(save_series_, [c_skill, d_pid, d_series, d_refresh], [d_proj, d_pid, d_refresh])

        @err
        def save_ep_(pid_, text):
            if not pid_:
                raise UserError("บันทึก series.json ก่อน")
            name, _ep = D.save_episode(pdir(pid_), text)
            save_project(get_project(pid_))
            gr.Info(f"บันทึก {name} แล้ว")
            return gr.update(choices=D.episodes(pdir(pid_)), value=name)

        d_save_ep.click(save_ep_, [d_pid, d_ep_text], d_ep).then(lambda r: r + 1, d_refresh, d_refresh)

        @err
        def check_(pid_, ep, sec):
            if not pid_ or not ep:
                raise UserError("เลือกตอนก่อน")
            _p, _t, root = _drama_template(pid_)
            ok, out = D.check(root, pdir(pid_), ep, int(sec or 0))
            (gr.Info if ok else gr.Warning)("บทผ่าน ✅" if ok else "บทยังมี ERROR ต้องแก้ก่อนเรนเดอร์ (WARN แค่คำแนะนำ)")
            return out

        d_check.click(check_, [d_pid, d_ep, d_sec], d_check_out)

        def _voice_cfg(p, engine, emodel):
            v = p.get("voices", {})
            return {"engine": engine, "model": emodel, "map": {k: x for k, x in v.get("map", {}).items() if x},
                    "default": v.get("default", "")}

        def _lib_dir(p):
            t = get_template(p.get("templateId")) or {"name": "unknown"}
            return DATA / "bg_library" / re.sub(r"[^\w-]+", "_", t["name"])

        def _place(pid_, plid):
            for pl in D.load(pdir(pid_), "series.json").get("places", []):
                if pl.get("id") == plid:
                    return pl
            raise UserError("ไม่พบสถานที่นี้ใน series.json")

        def _draw_one(pid_, plid, model, review, feedback, progress=None):
            p, _t, root = _drama_template(pid_)
            say = (lambda m: progress(0, desc=m[:70])) if progress else print
            D.draw_candidate(root, pdir(pid_), D.load(pdir(pid_), "series.json"), _place(pid_, plid), model,
                             review, feedback or "", say)
            gr.Info("วาดเสร็จ ดูภาพแล้วกด ✅ ถ้าโอเค")

        def _draw_many(pid_, places, model, review, progress=None):
            p, _t, root = _drama_template(pid_)
            series = D.load(pdir(pid_), "series.json")
            fails = []
            for k, pl in enumerate(places):
                if progress:
                    progress(k / len(places), desc=f"Claude วาด {pl.get('name', pl['id'])} ({k + 1}/{len(places)})")
                try:
                    D.draw_candidate(root, pdir(pid_), series, pl, model, review, "", print)
                except D.DramaError as e:
                    if "ANTHROPIC_API_KEY" in str(e) or "API key" in str(e):
                        raise
                    fails.append(f"{pl.get('name', pl['id'])}: {e}")
            if fails:
                gr.Warning("วาดไม่สำเร็จ: " + " | ".join(fails)[:400])
            gr.Info("วาดเสร็จ ดูแต่ละภาพแล้วกด ✅ ที่ฉากที่โอเค")

        def _approve(pid_, plid):
            p = get_project(pid_)
            D.approve_candidate(pdir(pid_), _place(pid_, plid), _lib_dir(p), p.get("title", ""))
            p = get_project(pid_)
            p.setdefault("bgs", {})[plid] = "claude"
            save_project(p)
            gr.Info("ใช้ฉากนี้แล้ว และเก็บเข้าคลังฉาก (ซีรีส์อื่นเลือกใช้ได้)")

        def _prep_bgs(p, pid_):
            d = pdir(pid_)
            series = D.load(d, "series.json")
            D.apply_backgrounds(d, series, p.get("bgs", {}), _lib_dir(p))
            miss = [x["place"].get("name", x["place"]["id"]) for x in D.bg_status(d, series, p.get("bgs", {}), _lib_dir(p))
                    if x["used"] and x["state"] == "missing"]
            if miss:
                gr.Warning(f"ยังไม่มีฉาก: {', '.join(miss)} → ใช้ฉากสำเร็จรูปแทน (ไปที่ขั้น 4 ให้ Claude วาดได้)")

        @err
        def still_(pid_, ep, n, engine, emodel):
            if not pid_ or not ep:
                raise UserError("เลือกตอนก่อน")
            p, _t, root = _drama_template(pid_)
            _prep_bgs(p, pid_)
            img = D.render(root, pdir(pid_), ep, _voice_cfg(p, engine, emodel), still=int(n or 1))
            return str(img)

        d_still.click(still_, [d_pid, d_ep, d_still_n, d_engine, d_emodel], d_img)

        @err
        def render_(pid_, ep, engine, emodel, force, progress=gr.Progress()):
            if not pid_ or not ep:
                raise UserError("เลือกตอนก่อน")
            p, _t, root = _drama_template(pid_)
            if engine == "eleven" and not os.environ.get("ELEVENLABS_API_KEY"):
                raise UserError("ยังไม่ได้ใส่ ELEVENLABS_API_KEY")
            ok, out = D.check(root, pdir(pid_), ep)
            if not ok and not force:
                raise UserError("บทยังไม่ผ่านการตรวจ กด ✅ ตรวจบท เพื่อดูว่าต้องแก้อะไร")
            log = []
            _prep_bgs(p, pid_)

            def on_line(line):
                log.append(line)
                if m := re.match(r"shot (\d+)/(\d+)", line):
                    progress(int(m.group(1)) / int(m.group(2)), desc=f"เรนเดอร์ช็อต {m.group(1)}/{m.group(2)}")
                elif m := re.match(r"voice (\d+)/(\d+)", line):
                    progress(0, desc=f"สร้างเสียง {m.group(1)}/{m.group(2)}")
                elif line.startswith("voice:"):
                    progress(0, desc="สร้างเสียงพากย์")

            progress(0, desc="เริ่ม")
            try:
                mp4 = D.render(root, pdir(pid_), ep, _voice_cfg(p, engine, emodel), on_line=on_line)
            except D.DramaError as e:
                silent = sorted(pdir(pid_).glob("ep*.mp4"))
                gr.Warning(str(e)[:300])
                return (str(silent[-1]) if silent and "ไม่มีเสียง" in str(e) else None), "\n".join(log[-60:])
            final = mp4.with_name(f"{mp4.stem}_{int(time.time())}.mp4")  # ชื่อใหม่ กัน browser cache
            for old in pdir(pid_).glob(f"{mp4.stem}_*.mp4"):
                old.unlink(missing_ok=True)
            shutil.copy(mp4, final)
            gr.Info("เรนเดอร์เสร็จ 🎉")
            return str(final), "\n".join(log[-60:])

        d_render.click(render_, [d_pid, d_ep, d_engine, d_emodel, d_force], [d_video, d_log])

        @err
        def d_del_(pid_, tid):
            if pid_:
                shutil.rmtree(pdir(pid_), ignore_errors=True)
            ch = _drama_choices(tid)
            return gr.update(choices=ch, value=None), "", "", ""

        d_del.click(d_del_, [d_pid, c_skill], [d_proj, d_pid, d_series, d_ep_text])

        # ===== events: settings
        @err
        def load_voices():
            _voice_cache[:] = list_voices()
            ch = _voice_choices()
            return f"โหลด {len(_voice_cache)} เสียงแล้ว", gr.update(choices=ch), gr.update(choices=ch)

        st_load.click(load_voices, None, [st_out, e_voice, s_voice])

        # โหลดรายชื่อเสียงอัตโนมัติถ้ามี key
        def on_start():
            if os.environ.get("ELEVENLABS_API_KEY") and not _voice_cache:
                try:
                    _voice_cache[:] = list_voices()
                except Exception:  # noqa: BLE001 — ไม่ให้หน้าเว็บพังเพราะโหลดเสียงไม่ได้
                    pass
            ch = _voice_choices()
            # รายชื่อเทมเพลต/โปรเจกต์ล่าสุดทุกครั้งที่เปิดหน้า (ไม่งั้นรีเฟรชแล้วเห็นของเก่า)
            return (gr.update(choices=ch), gr.update(choices=ch),
                    gr.update(choices=[("(ไม่ใช้เทมเพลต)", "")] + _template_choices()),
                    gr.update(choices=_template_choices()), gr.update(choices=_skill_choices()),
                    gr.update(choices=_project_choices()))

        app.load(on_start, None, [e_voice, s_voice, c_skill, s_pick, e_skill, e_proj])
        d_engine.change(err(lambda pid_, v: _set_engine(pid_, v)), [d_pid, d_engine], None)

        def _set_engine(pid_, v):
            if pid_:
                p = get_project(pid_)
                p.setdefault("voices", {})["engine"] = v
                save_project(p)

    return app


if __name__ == "__main__":
    build_app().queue().launch(share=bool(os.environ.get("COLAB_RELEASE_TAG")), allowed_paths=[str(DATA)])


In [ ]:
#@title 2. เปิดหน้าเว็บ (กดลิงก์ gradio.live ด้านล่าง)
import sys, importlib
sys.path.insert(0, '/content')
import clip_studio; importlib.reload(clip_studio)
clip_studio.build_app().queue().launch(share=True, debug=True, allowed_paths=[str(clip_studio.DATA)])